# RAG sobre los manuales CRP — comparable con `train_gemma.ipynb` / `evaluate_gemma.ipynb`

Mismo dataset (`qa.jsonl`) y **el mismo split de test** (`seed=42`,
`test_size=0.10`) que usa `evaluate_gemma.ipynb`, para que las respuestas de
este notebook sean directamente comparables contra el modelo **base** y el
modelo **afinado con LoRA** en la misma tabla final.

Pipeline:
1. Cargar `corpus_pages.jsonl` (páginas extraídas de los manuales CRP) y
   trocearlo en chunks más manejables.
2. Indexar los chunks con embeddings (`sentence-transformers`) + FAISS
   (CPU, no consume crédito de GPU — el índice se calcula una sola vez).
3. Para cada pregunta del split de test: recuperar los `k` chunks más
   relevantes, armar un prompt con contexto + pregunta, y generar la
   respuesta con `google/gemma-7b-it` **sin fine-tuning** (RAG puro).
4. Calcular ROUGE / BERTScore igual que en `evaluate_gemma.ipynb`.
5. Fusionar con el CSV que ya generó `evaluate_gemma.ipynb`
   (`eval_crobotp_base_vs_finetuned.csv`) para construir la tabla
   comparativa final: **base vs. afinado (LoRA) vs. RAG**.


## 0. Instalar dependencias

In [1]:
%pip install torch transformers accelerate bitsandbytes peft \
    sentence-transformers faiss-cpu rank_bm25 huggingface_hub \
    evaluate rouge_score bert_score absl-py pandas datasets


Note: you may need to restart the kernel to use updated packages.


## 0b. Login en Hugging Face

google/gemma-7b-it es un repo *gated* -necesitas un token de HF con la
licencia del modelo ya aceptada en tu cuenta (el mismo que usaste en
`train_gemma.ipynb`).


In [1]:
import getpass
from huggingface_hub import login

hf_token = getpass.getpass("Token de Hugging Face (con acceso a google/gemma-7b-it): ")
login(token=hf_token)
del hf_token  # no lo dejamos flotando en una variable mas de lo necesario


Token de Hugging Face (con acceso a google/gemma-7b-it):  ········


## 1. Configuración

Mismos valores de `DATA_PATH`, `EVAL_SPLIT_SIZE` y `SEED` que
`evaluate_gemma.ipynb` — así el split de test es *idéntico*, ejemplo por
ejemplo, al que se usó para comparar base vs. afinado.


In [2]:
import os

# --- Modelo generador (RAG usa el modelo BASE, sin LoRA) ---
MODEL_NAME = "google/gemma-7b-it"
MAX_NEW_TOKENS = 256

# --- Datos ---
CORPUS_PATH = "./corpus/corpus_pages.jsonl"   # manual, page, text
DATA_PATH = "./corpus/qa.jsonl"               # mismo archivo que el fine-tuning
EVAL_SPLIT_SIZE = 0.10                        # mismo 10% que evaluate_gemma.ipynb
SEED = 42                                     # misma semilla -> mismo split de test

# --- Chunking del corpus ---
CHUNK_SIZE = 800     # caracteres por chunk (algunas paginas tienen +30k chars)
CHUNK_OVERLAP = 100

# --- Retrieval ---
# BGE es un modelo entrenado para retrieval (no solo similitud generica de
# oraciones como MiniLM) -distingue mejor "un chunk que MENCIONA seguridad"
# de "el chunk que SI tiene el dato especifico que se pregunta".
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
# BGE recomienda anteponer esta instruccion SOLO a las preguntas (no a los
# chunks del corpus) para el caso de uso "pregunta corta -> pasaje largo".
QUERY_INSTRUCTION = "Represent this sentence for searching relevant passages: "
TOP_K = 6                 # paginas/chunks finales que entran al prompt
RETRIEVE_FETCH_K = 20      # candidatos por cada retriever antes de fusionar
NEIGHBOR_RADIUS = 1        # paginas vecinas a agregar alrededor de hits relevantes
NEIGHBOR_ANCHORS = 3       # solo los hits principales disparan la expansion
MAX_NEIGHBOR_PAGE_CHARS = 2500  # evita meter paginas enormes completas al prompt
MAX_CONTEXT_PAGES = 10     # limite total de paginas de contexto
DENSE_WEIGHT = 0.35        # peso del retrieval semantico
BM25_WEIGHT = 0.45         # peso del retrieval lexical
PHRASE_WEIGHT = 0.20       # peso de coincidencias literales de 3+ palabras

# --- Salidas ---
OUTPUT_CSV = os.environ.get("RAG_OUTPUT_CSV", "/home/jovyan/labs/eval_crobotp_rag.csv")
COMPARISON_CSV = os.environ.get(
    "COMPARISON_CSV", "/home/jovyan/labs/eval_crobotp_base_vs_finetuned_vs_rag.csv"
)
# CSV que ya produjo evaluate_gemma.ipynb (base vs. afinado), para fusionar al final
FINETUNE_EVAL_CSV = os.environ.get(
    "FINETUNE_EVAL_CSV", "/home/jovyan/labs/eval_crobotp_base_vs_finetuned.csv"
)

print(f"Modelo generador: {MODEL_NAME}")
print(f"Corpus: {CORPUS_PATH}")
print(f"QA: {DATA_PATH} (split test: {EVAL_SPLIT_SIZE}, seed={SEED})")
print(f"Modelo de embeddings: {EMBEDDING_MODEL}")


Modelo generador: google/gemma-7b-it
Corpus: ./corpus/corpus_pages.jsonl
QA: ./corpus/qa.jsonl (split test: 0.1, seed=42)
Modelo de embeddings: BAAI/bge-small-en-v1.5


**Nota sobre el retrieval (leer antes de correr la seccion 10 completa):**

Con `all-MiniLM-L6-v2` + `TOP_K=4` sin deduplicar, el retriever devolvia 2
chunks de la MISMA pagina y ademas confundia paginas que *hablan de*
seguridad con la pagina que *tiene* el dato especifico -se corrigio
deduplicando por `(manual, page)` y subiendo a `BAAI/bge-small-en-v1.5`.

Aun asi, algunas preguntas (ej. la del estado del motor) siguen fallando:
el manual tiene VARIAS paginas seguidas con el mismo estilo de "advertencia
de seguridad + consecuencia" (paginas 4-7 del Robot Operation Manual), asi
que para el embedding son casi indistinguibles entre si -todas hablan de
"seguridad" con la misma estructura. Por eso se agrega retrieval **hibrido**
(denso + BM25, fusionados con Reciprocal Rank Fusion):

- El componente **denso** (BGE) captura similitud semantica general.
- El componente **BM25** (lexico, por palabras clave) rescata coincidencias
  literales exactas -como "before operating the robot" + "motor" + "status",
  que SI aparecen tal cual en la pagina correcta y no en las paginas vecinas.

La implementación de abajo fusiona solo candidatos top-N de cada retriever y añade un pequeño bonus para coincidencias literales de frases técnicas. Vuelve a correr la sección 8 después de este cambio.

## 2. Verificar GPU\n\nSolo la necesitamos para *generar* con Gemma. El índice FAISS se construye en CPU.

In [3]:
import torch

print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Aviso: no se detecto GPU. Revisa el contenedor/VM (ver docker-compose.yml).")


CUDA disponible: True
GPU: NVIDIA L4


## 3. Cargar y trocear el corpus (`corpus_pages.jsonl`)

Cada registro del archivo es una página completa de un manual (`manual`,
`page`, `text`). Algunas páginas son muy largas (hay una con ~39.000
caracteres), así que las troceamos en chunks de `CHUNK_SIZE` caracteres con
solapamiento `CHUNK_OVERLAP`, para que el retriever recupere pasajes
puntuales en vez de páginas enteras.


In [4]:
import json

def load_corpus_pages(path):
    pages = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                pages.append(json.loads(line))
    return pages

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    text = text.strip()
    if len(text) <= chunk_size:
        return [text] if text else []
    chunks = []
    start = 0
    step = max(chunk_size - overlap, 1)
    while start < len(text):
        chunk = text[start:start + chunk_size].strip()
        if chunk:
            chunks.append(chunk)
        start += step
    return chunks

pages = load_corpus_pages(CORPUS_PATH)
print(f"Paginas cargadas: {len(pages)}")

chunks = []          # texto de cada chunk
chunk_meta = []       # (manual, page) de cada chunk, para citar la fuente
for p in pages:
    for c in chunk_text(p["text"]):
        chunks.append(c)
        chunk_meta.append({"manual": p["manual"], "page": p["page"]})

print(f"Chunks totales: {len(chunks)}")
print(f"Ejemplo de chunk:\n---\n{chunks[0][:300]}...\n---")

# Índice a nivel de página para expansión de contexto.
page_lookup = {}
page_order = {}
for p in pages:
    manual = p["manual"]
    page = int(p["page"])
    page_lookup[(manual, page)] = p["text"].strip()

for manual in sorted({m for m, _ in page_lookup}):
    page_order[manual] = sorted(page for (m, page) in page_lookup if m == manual)


Paginas cargadas: 227
Chunks totales: 695
Ejemplo de chunk:
---
Installation Dimension of Base
INSTALLATION INTERFACE DIAGRAM
MOTION RANGE DIAGRAM
Flange Dimensions
PRODUCT INTRODUCTION 39/40
CRP-RA07A-08 CRP-RA09A-07
CRP-RA07A-08
CRP-RA09A-07
INDUSTRIAL ROBOT
HANDLING 
APPLICATION
· Adopt modular design to effectively reduce the failure rate of the whole machin...
---


## 4. Embeddings + índice FAISS

Se calcula una sola vez (CPU, no requiere GPU ni gasta crédito de Vertex).


In [5]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBEDDING_MODEL)

chunk_embeddings = embedder.encode(
    chunks, batch_size=64, show_progress_bar=True, normalize_embeddings=True,
)
chunk_embeddings = np.asarray(chunk_embeddings, dtype="float32")

dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # producto interno sobre vectores normalizados = similitud coseno
index.add(chunk_embeddings)

print(f"Indice FAISS construido: {index.ntotal} vectores de dimension {dim}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Indice FAISS construido: 695 vectores de dimension 384


## 4b. Índice BM25 (retrieval híbrido)

BM25 es búsqueda léxica clásica (como la de un buscador de texto completo):
no entiende semántica, pero es muy fuerte encontrando coincidencias
*literales* de palabras clave -exactamente lo que falla en un embedding
cuando varias páginas comparten el mismo tema y estructura.


In [6]:
import re
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(r"[a-zA-Z0-9]+", text.lower())

tokenized_chunks = [tokenize(c) for c in chunks]
bm25 = BM25Okapi(tokenized_chunks)

print(f"Indice BM25 construido sobre {len(tokenized_chunks)} chunks.")


Indice BM25 construido sobre 695 chunks.


## 5. Función de recuperación (retrieval)

In [7]:
import numpy as np
import re

def top_indices(scores, n):
    """Devuelve los índices de los n candidatos con mayor score."""
    n = min(int(n), len(scores))
    if n <= 0:
        return np.array([], dtype=int)
    return np.argsort(-scores)[:n]

def minmax_normalize(values):
    """Normaliza un vector a [0,1] sin romperse si todos los valores son iguales."""
    values = np.asarray(values, dtype=np.float32)
    if len(values) == 0:
        return values
    lo, hi = float(values.min()), float(values.max())
    if hi - lo < 1e-8:
        return np.ones_like(values)
    return (values - lo) / (hi - lo)

def longest_query_ngram_in_text(query, text, min_n=3, max_n=6):
    """
    Busca la mayor secuencia consecutiva de palabras de la pregunta
    que aparece literalmente en el chunk. Esto ayuda mucho en manuales
    técnicos, donde frases como 'before operating the robot' son muy
    informativas.
    """
    q_tokens = tokenize(query)
    text_norm = " ".join(tokenize(text))

    max_n = min(max_n, len(q_tokens))
    for n in range(max_n, min_n - 1, -1):
        for start in range(len(q_tokens) - n + 1):
            phrase = " ".join(q_tokens[start:start+n])
            if phrase in text_norm:
                return n
    return 0

def retrieve(query, k=TOP_K, fetch_k=RETRIEVE_FETCH_K):
    """
    Retrieval híbrido:
      1) BGE para similitud semántica
      2) BM25 para coincidencias léxicas
      3) se conserva la unión de los top-fetch_k de ambos retrievers
      4) se reranquea con scores normalizados + bonus de frases literales
      5) se deduplica por (manual, página)
    """
    # --- Dense retrieval (BGE) ---
    q_text = QUERY_INSTRUCTION + query
    q_emb = embedder.encode(
        [q_text],
        normalize_embeddings=True
    ).astype("float32")[0]

    dense_scores = chunk_embeddings @ q_emb

    # --- Lexical retrieval (BM25) ---
    bm25_scores = np.asarray(bm25.get_scores(tokenize(query)), dtype=np.float32)

    # IMPORTANTE: no fusionamos los 695 rankings completos.
    # Fusionamos solo los mejores candidatos de cada retriever.
    dense_top = top_indices(dense_scores, fetch_k)
    bm25_top = top_indices(bm25_scores, fetch_k)

    candidates = np.array(
        sorted(set(dense_top.tolist()) | set(bm25_top.tolist())),
        dtype=int
    )

    # Normalización dentro de la unión de candidatos.
    dense_norm = np.zeros(len(candidates), dtype=np.float32)
    bm25_norm = np.zeros(len(candidates), dtype=np.float32)
    if len(candidates):
        dense_norm = minmax_normalize(dense_scores[candidates])
        bm25_norm = minmax_normalize(bm25_scores[candidates])

    candidate_results = []
    for pos, idx in enumerate(candidates):
        ngram_len = longest_query_ngram_in_text(query, chunks[idx])
        phrase_score = min(max(ngram_len - 2, 0), 4) / 4.0  # 3-gram=0.25 ... 6-gram=1.0

        hybrid_score = (
            DENSE_WEIGHT * float(dense_norm[pos])
            + BM25_WEIGHT * float(bm25_norm[pos])
            + PHRASE_WEIGHT * phrase_score
        )

        candidate_results.append({
            "idx": int(idx),
            "hybrid_score": hybrid_score,
            "dense_score": float(dense_scores[idx]),
            "bm25_score": float(bm25_scores[idx]),
            "phrase_score": phrase_score,
        })

    candidate_results.sort(key=lambda x: x["hybrid_score"], reverse=True)

    # --- Deduplicar por página ---
    seen_pages = set()
    results = []

    for item in candidate_results:
        i = item["idx"]
        key = (chunk_meta[i]["manual"], chunk_meta[i]["page"])

        if key in seen_pages:
            continue

        seen_pages.add(key)
        results.append({
            "text": chunks[i],
            "manual": chunk_meta[i]["manual"],
            "page": chunk_meta[i]["page"],
            "score": float(item["hybrid_score"]),
            "dense_score": item["dense_score"],
            "bm25_score": item["bm25_score"],
            "phrase_score": item["phrase_score"],
        })

        if len(results) >= k:
            break

    return results

# Sanity check: esta pregunta tiene una coincidencia literal muy fuerte en
# EN-00001-A1_Robot_Operation_Manual__1 p.5 y en CRP_programming_instruction p.3.
_ejemplo = retrieve("What safety status must the motor be in before operating the robot?")

for rank, r in enumerate(_ejemplo, 1):
    print(
        f"{rank}. [{r['manual']} p.{r['page']} | "
        f"hybrid={r['score']:.4f} | dense={r['dense_score']:.3f} | "
        f"bm25={r['bm25_score']:.2f} | phrase={r['phrase_score']:.2f}]"
    )
    print(f"   {r['text'][:250].replace(chr(10), ' ')}...")


def expand_with_neighbors(
    retrieved,
    radius=NEIGHBOR_RADIUS,
    max_pages=MAX_CONTEXT_PAGES,
    anchor_count=NEIGHBOR_ANCHORS,
):
    """
    Añade paginas vecinas al contexto sin alterar el ranking del retrieval.

    Para procedimientos tecnicos, prioriza las paginas del mismo manual que
    contiene el hit mejor rankeado. Esto ayuda a reconstruir secuencias como
    p.20 -> p.21 -> p.22 -> p.23 sin llenar el limite con otros manuales.
    """
    if not retrieved:
        return []

    # Hits directos, deduplicados por pagina.
    selected = {}
    for rank, r in enumerate(retrieved, 1):
        key = (r["manual"], int(r["page"]))
        selected.setdefault(key, {
            "manual": r["manual"],
            "page": int(r["page"]),
            "text": r["text"],
            "score": r["score"],
            "source": "retrieved",
            "retrieval_rank": rank,
        })

    if len(selected) >= max_pages or not retrieved:
        return list(selected.values())[:max_pages]

    # Los anchors salen primero del manual del hit #1. Si quedan menos de
    # anchor_count, completamos con otros hits de otros manuales.
    primary_manual = retrieved[0]["manual"]
    anchors = [r for r in retrieved if r["manual"] == primary_manual][:max(0, int(anchor_count))]
    if len(anchors) < anchor_count:
        seen = {(r["manual"], int(r["page"])) for r in anchors}
        for r in retrieved:
            key = (r["manual"], int(r["page"]))
            if key not in seen:
                anchors.append(r)
                seen.add(key)
            if len(anchors) >= anchor_count:
                break

    # Generar vecinos candidatos y quedarnos primero con los mas cercanos.
    neighbor_candidates = {}
    for anchor_rank, r in enumerate(anchors, 1):
        manual = r["manual"]
        page = int(r["page"])
        ordered_pages = page_order.get(manual, [])
        if page not in ordered_pages:
            continue

        pos = ordered_pages.index(page)
        for d in range(1, int(radius) + 1):
            for neighbor_pos in (pos - d, pos + d):
                if not (0 <= neighbor_pos < len(ordered_pages)):
                    continue
                neighbor_page = ordered_pages[neighbor_pos]
                key = (manual, neighbor_page)
                if key in selected:
                    continue

                candidate = {
                    "manual": manual,
                    "page": neighbor_page,
                    "distance": d,
                    "anchor_rank": anchor_rank,
                    "anchor_score": float(r["score"]),
                    "text": page_lookup[key],
                }
                old = neighbor_candidates.get(key)
                priority = (candidate["distance"], candidate["anchor_rank"], -candidate["anchor_score"])
                old_priority = None if old is None else (old["distance"], old["anchor_rank"], -old["anchor_score"])
                if old is None or priority < old_priority:
                    neighbor_candidates[key] = candidate

    neighbor_items = sorted(
        neighbor_candidates.values(),
        key=lambda x: (x["distance"], x["anchor_rank"], -x["anchor_score"])
    )

    # Añadir vecinos hasta llenar el presupuesto.
    for n in neighbor_items:
        if len(selected) >= max_pages:
            break
        text = n["text"]
        if len(text) > MAX_NEIGHBOR_PAGE_CHARS:
            text = text[:MAX_NEIGHBOR_PAGE_CHARS].rstrip() + "\n[Neighbor page truncated]"
        key = (n["manual"], n["page"])
        selected[key] = {
            "manual": n["manual"],
            "page": n["page"],
            "text": text,
            "score": n["anchor_score"] - 0.01 * n["distance"],
            "source": f"neighbor_{n['distance']}",
            "retrieval_rank": None,
        }

    # Orden de lectura: primero el manual del hit principal, y dentro de cada
    # manual las paginas en orden ascendente.
    manual_first_rank = {}
    for rank, x in enumerate(retrieved, 1):
        manual_first_rank.setdefault(x["manual"], rank)

    return sorted(
        selected.values(),
        key=lambda r: (manual_first_rank[r["manual"]], int(r["page"]))
    )[:max_pages]



1. [EN-00001-A1_Robot_Operation_Manual__1 p.5 | hybrid=0.7537 | dense=0.705 | bm25=23.49 | phrase=0.50]
   Safety Attentions ★Before operating the robot, press the emergency stop button on the  teaching pendant and make sure main power supply of servo is off and  motor is in "off-power and brake"status. After cutting off the servo power,  servo power indi...
2. [EN-00001-A1_Robot_Operation_Manual__1 p.7 | hybrid=0.7409 | dense=0.760 | bm25=23.02 | phrase=0.00]
   ★All the operators of robot systems should participate in system  training, study safety protection measurements and understand robot  functionality. ★Make sure no abnormal or dangerous situation of robot and  auxiliary equipment occurs, before start...
3. [CRP_programming_instruction p.5 | hybrid=0.7340 | dense=0.759 | bm25=22.83 | phrase=0.00]
   Page: 5 of 72 Safe operation protocols ★All the operators of robot systems should participate in system training, study safety protection  measurements and understand robot function

## 6. Cargar el modelo generador (`google/gemma-7b-it`, base, sin LoRA)

Mismo `BitsAndBytesConfig` (4-bit) que `train_gemma.ipynb` /
`evaluate_gemma.ipynb`, para que el costo de cómputo y la calidad de
generación sean comparables.


In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Modelo base cargado (sin adaptadores LoRA).")
print("Dispositivo principal del modelo:", model.device)


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Modelo base cargado (sin adaptadores LoRA).
Dispositivo principal del modelo: cuda:0


## 7. Prompt RAG + función de generación

El contexto recuperado se antepone a la pregunta dentro del mismo mensaje
`user` (la plantilla de chat de Gemma no tiene rol `system`).


In [9]:
RAG_INSTRUCTION = (
    "You are a technical assistant for CRP robot manuals.\n"
    "Answer the question using ONLY the supplied context.\n"
    "Prefer the passages that directly answer the question.\n"
    "When the answer is a procedure, use the relevant consecutive pages to "
    "reconstruct the steps in the correct order.\n"
    "Keep technical terminology and operating states exactly as written "
    "when the context gives an exact term.\n"
    "Do not invent information or combine unrelated passages.\n"
    "If the context does not contain the answer, say: "
    "\"I don't know based on the provided context.\"\n\n"
)


def build_rag_prompt(question, k=TOP_K, neighbor_radius=NEIGHBOR_RADIUS, max_context_pages=MAX_CONTEXT_PAGES):
    retrieved = retrieve(question, k=k)
    context_items = expand_with_neighbors(
        retrieved,
        radius=neighbor_radius,
        max_pages=max_context_pages,
    )

    if not context_items:
        context = "(No relevant context was retrieved.)"
    else:
        context = "\n\n".join(
            f"[Source {i}: {r['manual']}, page {r['page']}, {r['source']}]\n{r['text']}"
            for i, r in enumerate(context_items, 1)
        )

    user_content = (
        f"{RAG_INSTRUCTION}"
        f"CONTEXT:\n{context}\n\n"
        f"QUESTION: {question}\n"
        f"ANSWER:"
    )

    return user_content, retrieved, context_items


def responder_rag(question, max_new_tokens=MAX_NEW_TOKENS, k=TOP_K, neighbor_radius=NEIGHBOR_RADIUS, max_context_pages=MAX_CONTEXT_PAGES):
    user_content, retrieved, context_items = build_rag_prompt(
        question,
        k=k,
        neighbor_radius=neighbor_radius,
        max_context_pages=max_context_pages,
    )

    messages = [{"role": "user", "content": user_content}]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    input_device = next(model.parameters()).device
    inputs = {key: value.to(input_device) for key, value in inputs.items()}

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    new_tokens = output_ids[0, input_len:]

    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    return answer, retrieved, context_items


## 8. Prueba rápida manual (2-3 preguntas)

In [10]:
test_questions = [
    "What safety status must the motor be in before operating the robot?",
    "How do you establish the tool coordinate system?",
]

for q in test_questions:
    answer, retrieved, context_items = responder_rag(q)

    print("=" * 100)
    print(f"Q: {q}")

    print("\nHits directos del retrieval:")
    for rank, r in enumerate(retrieved, 1):
        print(
            f"  {rank}. {r['manual']} p.{r['page']} | "
            f"hybrid={r['score']:.4f} | "
            f"dense={r['dense_score']:.3f} | "
            f"bm25={r['bm25_score']:.2f} | "
            f"phrase={r['phrase_score']:.2f}"
        )

    print("\nContexto final enviado a Gemma:")
    for rank, r in enumerate(context_items, 1):
        print(f"  {rank}. {r['manual']} p.{r['page']} | {r['source']}")

    print(f"\nA: {answer}")


Q: What safety status must the motor be in before operating the robot?

Hits directos del retrieval:
  1. EN-00001-A1_Robot_Operation_Manual__1 p.5 | hybrid=0.7537 | dense=0.705 | bm25=23.49 | phrase=0.50
  2. EN-00001-A1_Robot_Operation_Manual__1 p.7 | hybrid=0.7409 | dense=0.760 | bm25=23.02 | phrase=0.00
  3. CRP_programming_instruction p.5 | hybrid=0.7340 | dense=0.759 | bm25=22.83 | phrase=0.00
  4. CRP_programming_instruction p.3 | hybrid=0.7198 | dense=0.719 | bm25=21.22 | phrase=0.50
  5. EN-00001-A1_Robot_Operation_Manual__1 p.27 | hybrid=0.6959 | dense=0.760 | bm25=21.32 | phrase=0.00
  6. EN-00001-A1_Robot_Operation_Manual__1 p.12 | hybrid=0.6485 | dense=0.745 | bm25=20.49 | phrase=0.00

Contexto final enviado a Gemma:
  1. EN-00001-A1_Robot_Operation_Manual__1 p.4 | neighbor_1
  2. EN-00001-A1_Robot_Operation_Manual__1 p.5 | retrieved
  3. EN-00001-A1_Robot_Operation_Manual__1 p.6 | neighbor_1
  4. EN-00001-A1_Robot_Operation_Manual__1 p.7 | retrieved
  5. EN-00001-A1_Robot

### Diagnóstico esperado de la sección 8

Para la pregunta del estado del motor, el retrieval debe incluir `EN-00001-A1_Robot_Operation_Manual__1`, página 5, o `CRP_programming_instruction`, página 3. Esas páginas contienen literalmente el estado `off-power and brake`.

Si esas páginas aparecen entre las fuentes pero la respuesta sigue diciendo que no sabe, el problema ya no es retrieval sino prompt/model generation. Si no aparecen, el problema está en el índice/retrieval.


La expansión de contexto agrega páginas vecinas del mismo manual alrededor de los hits principales (los tres primeros hits por defecto). Esto es especialmente útil para procedimientos distribuidos en varias paginas consecutivas.

**Importante:** las paginas vecinas no reemplazan los hits directos del retrieval; solo amplían el contexto que ve Gemma.


En una consulta procedimental, el contexto final puede contener la pagina recuperada y sus paginas vecinas del mismo manual. Por ejemplo, una consulta centrada en p.23 puede incorporar p.20–p.24 para conservar los pasos del procedimiento.


In [11]:
print("EMBEDDING_MODEL configurado:", EMBEDDING_MODEL)
print(embedder)
print("chunk_embeddings shape:", chunk_embeddings.shape)
print("TOP_K:", TOP_K, "| QUERY_INSTRUCTION:", QUERY_INSTRUCTION)

EMBEDDING_MODEL configurado: BAAI/bge-small-en-v1.5
SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'cls', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)
chunk_embeddings shape: (695, 384)
TOP_K: 6 | QUERY_INSTRUCTION: Represent this sentence for searching relevant passages: 


## 9. Cargar el MISMO split de test que `evaluate_gemma.ipynb`

Se reconstruye con `train_test_split(test_size=EVAL_SPLIT_SIZE, seed=SEED)`
sobre `qa.jsonl` — con la misma semilla, `datasets` devuelve exactamente el
mismo split que se usó para evaluar base vs. afinado.


In [12]:
from datasets import load_dataset

full_dataset = load_dataset("json", data_files=DATA_PATH, split="train")
dataset_split = full_dataset.train_test_split(test_size=EVAL_SPLIT_SIZE, seed=SEED)
test_dataset = dataset_split["test"]

prompts = [ex["messages"][0]["content"] for ex in test_dataset]
references = [ex["messages"][1]["content"] for ex in test_dataset]

print(f"Evaluando RAG sobre {len(test_dataset)} ejemplos (mismo split que evaluate_gemma.ipynb).")


Evaluando RAG sobre 98 ejemplos (mismo split que evaluate_gemma.ipynb).


## 10. Generar respuestas RAG para todo el split de test

In [13]:
preds_rag = []
sources_rag = []
context_sources_rag = []

print("Generando respuestas con RAG (modelo base + contexto recuperado)...")
for i, q in enumerate(prompts, 1):
    answer, retrieved, context_items = responder_rag(q)
    preds_rag.append(answer)
    sources_rag.append("; ".join(f"{r['manual']} p.{r['page']}" for r in retrieved))
    context_sources_rag.append("; ".join(f"{r['manual']} p.{r['page']}" for r in context_items))
    if i % 10 == 0 or i == len(prompts):
        print(f"  ... RAG: {i}/{len(prompts)}")


Generando respuestas con RAG (modelo base + contexto recuperado)...
  ... RAG: 10/98
  ... RAG: 20/98
  ... RAG: 30/98
  ... RAG: 40/98
  ... RAG: 50/98
  ... RAG: 60/98
  ... RAG: 70/98
  ... RAG: 80/98
  ... RAG: 90/98
  ... RAG: 98/98


## 11. Calcular ROUGE y BERTScore (igual que `evaluate_gemma.ipynb`)

Nota: `evaluate_gemma.ipynb` usa `lang="es"` en BERTScore, pero el dataset
(`qa.jsonl`) está en inglés — probablemente un residuo de copiar la
plantilla del lab de resumen en español. Aquí usamos `lang="en"` (deja que
`evaluate` elija su modelo por defecto para inglés, más preciso que forzar
uno multilingüe). Para que la comparación final sea justa, en la sección 13
recalculamos BERTScore para base/afinado con esta MISMA configuración, en
vez de reusar los números de `evaluate_gemma.ipynb`.


In [14]:
import evaluate

BERTSCORE_LANG = "en"

rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

rouge_rag = rouge.compute(predictions=preds_rag, references=references)
rouge_rag_per_example = rouge.compute(predictions=preds_rag, references=references, use_aggregator=False)

bs_rag = bertscore.compute(predictions=preds_rag, references=references, lang=BERTSCORE_LANG)
bertscore_f1_rag = sum(bs_rag["f1"]) / len(bs_rag["f1"])

print("ROUGE (RAG):     ", rouge_rag)
print("BERTScore F1 (RAG):", round(bertscore_f1_rag, 4))


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ROUGE (RAG):      {'rouge1': np.float64(0.13793129663751527), 'rouge2': np.float64(0.0535310318846753), 'rougeL': np.float64(0.12444524928455551), 'rougeLsum': np.float64(0.12370878459338278)}
BERTScore F1 (RAG): 0.8346


## 12. Guardar el detalle por ejemplo (RAG)

In [16]:
import pandas as pd

df_rag = pd.DataFrame({
    "pregunta": prompts,
    "referencia": references,
    "respuesta_rag": preds_rag,
    "fuentes_rag": sources_rag,
    "contexto_rag": context_sources_rag,
    "rougeL_rag": rouge_rag_per_example["rougeL"],
})
df_rag.to_csv(OUTPUT_CSV, index=False)
print(f"Detalle RAG guardado en: {OUTPUT_CSV}")


Detalle RAG guardado en: /home/jovyan/labs/eval_crobotp_rag.csv


## 13. Tabla comparativa final: base vs. afinado (LoRA) vs. RAG

Se fusiona con `eval_crobotp_base_vs_finetuned.csv` (el CSV que produce
`evaluate_gemma.ipynb`), haciendo match por la columna `pregunta`. Como ese
CSV solo guardó `rougeL` por ejemplo (no ROUGE completo ni BERTScore),
aquí se recalculan ROUGE y BERTScore para `respuesta_base` y
`respuesta_afinada` con la MISMA configuración que usamos para RAG —así
las tres columnas de la tabla final son comparables entre sí, sin depender
de qué `lang`/modelo se haya usado quizás distinto en el otro notebook.


In [17]:
import os

if not os.path.isfile(FINETUNE_EVAL_CSV):
    print(f"No se encontro {FINETUNE_EVAL_CSV}.")
    print("Corre evaluate_gemma.ipynb primero para poder comparar base/afinado/RAG.")
    print("Mientras tanto, aqui estan los resultados de RAG solo:")
    df_resumen_rag = pd.DataFrame({
        "modelo": ["RAG (base + contexto recuperado)"],
        "rouge1": [rouge_rag["rouge1"]],
        "rouge2": [rouge_rag["rouge2"]],
        "rougeL": [rouge_rag["rougeL"]],
        "rougeLsum": [rouge_rag["rougeLsum"]],
        "bertscore_f1": [bertscore_f1_rag],
        "longitud_promedio_palabras": [sum(len(p.split()) for p in preds_rag) / len(preds_rag)],
    })
else:
    df_ft = pd.read_csv(FINETUNE_EVAL_CSV)

    # Fusionar por pregunta (misma columna en ambos notebooks, mismo split -> deberia calzar 1 a 1)
    df_merged = df_rag.merge(df_ft, on=["pregunta", "referencia"], how="inner")
    print(f"Ejemplos fusionados (deberia ser ~{len(df_rag)}): {len(df_merged)}")
    if len(df_merged) != len(df_rag):
        print("Aviso: no todos los ejemplos calzaron -verifica que ambos notebooks usen "
              "el mismo DATA_PATH, EVAL_SPLIT_SIZE y SEED.")

    # Recalcular ROUGE + BERTScore para base y afinado con la misma config que RAG
    rouge_base_full = rouge.compute(predictions=df_merged["respuesta_base"].tolist(), references=df_merged["referencia"].tolist())
    rouge_ft_full = rouge.compute(predictions=df_merged["respuesta_afinada"].tolist(), references=df_merged["referencia"].tolist())
    rouge_rag_full = rouge.compute(predictions=df_merged["respuesta_rag"].tolist(), references=df_merged["referencia"].tolist())

    bs_base = bertscore.compute(predictions=df_merged["respuesta_base"].tolist(), references=df_merged["referencia"].tolist(), lang=BERTSCORE_LANG)
    bs_ft = bertscore.compute(predictions=df_merged["respuesta_afinada"].tolist(), references=df_merged["referencia"].tolist(), lang=BERTSCORE_LANG)
    bs_rag_full = bertscore.compute(predictions=df_merged["respuesta_rag"].tolist(), references=df_merged["referencia"].tolist(), lang=BERTSCORE_LANG)

    def avg_len(col):
        return sum(len(x.split()) for x in df_merged[col]) / len(df_merged)

    df_resumen_rag = pd.DataFrame({
        "modelo": ["base (sin LoRA, sin RAG)", "afinado (LoRA)", "RAG (base + contexto)"],
        "rouge1": [rouge_base_full["rouge1"], rouge_ft_full["rouge1"], rouge_rag_full["rouge1"]],
        "rouge2": [rouge_base_full["rouge2"], rouge_ft_full["rouge2"], rouge_rag_full["rouge2"]],
        "rougeL": [rouge_base_full["rougeL"], rouge_ft_full["rougeL"], rouge_rag_full["rougeL"]],
        "rougeLsum": [rouge_base_full["rougeLsum"], rouge_ft_full["rougeLsum"], rouge_rag_full["rougeLsum"]],
        "bertscore_f1": [
            sum(bs_base["f1"]) / len(bs_base["f1"]),
            sum(bs_ft["f1"]) / len(bs_ft["f1"]),
            sum(bs_rag_full["f1"]) / len(bs_rag_full["f1"]),
        ],
        "longitud_promedio_palabras": [
            avg_len("respuesta_base"), avg_len("respuesta_afinada"), avg_len("respuesta_rag"),
        ],
    })

    df_merged.to_csv(COMPARISON_CSV, index=False)
    print(f"Detalle fusionado (base/afinado/RAG por ejemplo) guardado en: {COMPARISON_CSV}")

referencia_len = sum(len(r.split()) for r in references) / len(references)
print(f"(Longitud promedio de la referencia: {referencia_len:.1f} palabras)")
df_resumen_rag


No se encontro /home/jovyan/labs/eval_crobotp_base_vs_finetuned.csv.
Corre evaluate_gemma.ipynb primero para poder comparar base/afinado/RAG.
Mientras tanto, aqui estan los resultados de RAG solo:
(Longitud promedio de la referencia: 7.2 palabras)


,modelo,rouge1,rouge2,rougeL,rougeLsum,bertscore_f1,longitud_promedio_palabras
0,RAG (base + contexto recuperado),0.137931,0.053531,0.124445,0.123709,0.834597,25.969388


## Notas finales

- **Por qué el mismo split:** usar `seed=42` y `test_size=0.10` idénticos
  garantiza que las 977 preguntas de `qa.jsonl` se dividen exactamente
  igual en los tres notebooks, así que las tres filas de la tabla final
  responden a las mismas preguntas sobre las mismas referencias.
- **RAG vs. fine-tuning, qué esperar:** el fine-tuning le enseña al modelo
  el *estilo* de respuesta del dataset (formato, brevedad, tono), mientras
  que el RAG le da *hechos frescos* del corpus sin tocar los pesos del
  modelo. Es normal que RAG gane en preguntas muy específicas de un manual
  (números de parámetro, nombres de instrucciones exactas) y pierda un poco
  en ROUGE si el estilo de redacción del modelo base no calza tan bien con
  las referencias como el modelo afinado.
- **Costo:** el índice FAISS se construye en CPU (gratis); solo la
  generación con Gemma-7b-it usa la GPU L4 que ya tenías reservada para el
  fine-tuning -reutiliza la misma VM/contenedor, no necesitas una segunda.
- **Ampliación opcional:** se puede repetir este mismo notebook cargando el
  modelo **afinado + RAG** (adaptadores LoRA + contexto recuperado) para
  ver si ambas técnicas se complementan -bastaría con cargar el modelo como
  en `evaluate_gemma.ipynb` (con `PeftModel.from_pretrained`) en vez del
  modelo base de la sección 6, y correr el resto del notebook igual.
